# Linear support vector machines: affine scores and maximum margins

**Lecture 11 · Notebook 01 · Core route: 100 minutes · Full laboratory: 160 minutes**

A linear support vector machine uses the familiar score $z=w^\top x+b$, but learns it through a margin objective. We will develop its geometry, hinge loss, regularization, subgradients, and evaluation before deciding what it truly shares with a single neuron.

## Learning objectives

### How to use this notebook

You will learn to distinguish representation from objective, derive hyperplane distance and margin width, explain hard and soft margins, calculate a hinge-loss subgradient, fit a linear SVM on versioned real data, and compare three configurations of a single-neuron model.

Predict each example’s margin status before executing code. The core route ends after held-out evaluation; derivation and implementation exercises form the practice route.

## 1. Why this matters: one neuron, several learning problems

A linear SVM can be expressed with the same **single-neuron architecture** used for linear and logistic regression. All three begin with the affine preactivation

$$
z(x)=w^\top x+b
$$

The crucial lesson is that an architecture does not by itself define a model. The activation, loss, regularization, and output interpretation complete the specification:

- identity activation with squared loss gives linear regression;
- sigmoid activation with log loss gives logistic regression and a conditional probability;
- identity activation with regularized hinge loss gives a linear SVM score and rewards a wide margin.

Thus the models reuse a single-neuron computation without becoming equivalent estimators. For now, keep the mathematical pieces separate: affine computation, activation, objective, regularization, optimization method, and output meaning. The next notebook will ask how software should represent that shared structure.

In [ ]:
from __future__ import annotations

import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from rice_dsm.ml import LinearSVM
from rice_dsm.paths import course_database_path

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 438

## 2. Build the geometry from first principles

### 2.1 A hyperplane is a level set with a normal direction

Begin with the affine preactivation $z(x)=w^\top x+b$. Holding $z$ fixed produces a level set. In two dimensions, $z(x)=0$ is a line; in $p$ dimensions it is the hyperplane

$$H=\{x\in\mathbb R^p:w^\top x+b=0\}. $$

Why is $w$ normal to $H$? If $u$ and $v$ are any two points on $H$, then

$$w^\top u+b=0,\qquad w^\top v+b=0,$$

so subtraction gives $w^\top(u-v)=0$. Every direction $u-v$ lying along the hyperplane is perpendicular to $w$. The score therefore changes fastest in the $w$ direction.

Now take a point $q$ away from $H$ and let $q_H$ be its orthogonal projection onto $H$. Because the shortest route is parallel to the normal, write $q-q_H=t w$. Substituting $q_H=q-tw$ into $w^\top q_H+b=0$ gives

$$t=\frac{w^\top q+b}{\lVert w\rVert_2^2}.$$

Hence the signed distance and ordinary distance are

$$d_{\pm}(q,H)=\frac{w^\top q+b}{\lVert w\rVert_2},\qquad d(q,H)=\frac{|w^\top q+b|}{\lVert w\rVert_2}.$$

The next cell draws every object in this derivation. Change $w$, $b$, or $q$ and predict what must move before rerunning it.

In [ ]:
# A completely visible two-dimensional hyperplane: 2 x_1 - x_2 - 1 = 0.
w_geometry = np.array([2.0, -1.0])
b_geometry = -1.0
q = np.array([2.0, 1.5])

score_q = float(q @ w_geometry + b_geometry)
q_projection = q - score_q / (w_geometry @ w_geometry) * w_geometry
signed_distance = score_q / np.linalg.norm(w_geometry)

horizontal = np.linspace(-0.5, 2.8, 300)
boundary = (w_geometry[0] * horizontal + b_geometry) / (-w_geometry[1])

fig, axis = plt.subplots(figsize=(8, 6))
axis.plot(horizontal, boundary, color="black", linewidth=2.4,
          label=r"hyperplane $H: w^T x+b=0$")
axis.scatter(*q, s=80, color="tab:red", zorder=4, label=r"point $q$")
axis.scatter(*q_projection, s=75, color="tab:green", zorder=4,
             label=r"projection $q_H$")
axis.plot([q[0], q_projection[0]], [q[1], q_projection[1]],
          color="tab:red", linestyle="--", linewidth=2,
          label=r"shortest displacement $q-q_H$")
normal_origin = np.array([1.0, 1.0])
axis.annotate("", xy=normal_origin + 0.65 * w_geometry, xytext=normal_origin,
              arrowprops={"arrowstyle": "->", "lw": 2.2,
                          "color": "tab:blue"})
axis.text(*(normal_origin + 0.68 * w_geometry), r"normal $w$",
          color="tab:blue", fontsize=11)
distance_label = (f"$z(q)={score_q:.1f}$\n"
                  + rf"$d(q,H)=|z(q)|/\|w\|={abs(signed_distance):.3f}$")
axis.text(-0.28, 3.15, distance_label,
          bbox={"facecolor": "white", "alpha": 0.9, "edgecolor": "0.75"})
axis.set(xlabel=r"$x_1$", ylabel=r"$x_2$",
         title="The affine score creates a hyperplane and its normal geometry",
         xlim=(-0.5, 3.3), ylim=(-2.0, 4.0), aspect="equal")
axis.legend(loc="lower right", fontsize=9)
plt.tight_layout()

assert np.isclose(q_projection @ w_geometry + b_geometry, 0.0)
assert np.allclose(q - q_projection,
                   score_q / (w_geometry @ w_geometry) * w_geometry)

### 2.2 Functional margin is not geometric distance

For a signed label $y_i\in\{-1,+1\}$, define the **functional margin** $m_i=y_i z_i$. It answers two questions: is the sign correct, and how large is the raw score? But raw score depends on an arbitrary parameter scale. Replacing $(w,b)$ by $(c w,c b)$ for any $c>0$ leaves the boundary and every class decision unchanged while multiplying $m_i$ by $c$.

Dividing by $\lVert w\rVert_2$ removes that arbitrariness. The **geometric signed margin** is

$$\gamma_i=\frac{y_i(w^\top x_i+b)}{\lVert w\rVert_2}.$$

It is the signed Euclidean distance to the decision hyperplane. The computation below verifies the distinction instead of asking you to accept it.

In [ ]:
scales = np.array([0.5, 1.0, 2.0, 4.0])
label_q = 1.0
functional_margins = label_q * scales * score_q
geometric_margins = functional_margins / (scales * np.linalg.norm(w_geometry))

invariance_table = pd.DataFrame({
    "scale c": scales,
    "functional margin y(cz)": functional_margins,
    "geometric margin y(cz)/||cw||": geometric_margins,
})
display(invariance_table)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for scale, color in zip(scales, ["0.75", "tab:blue", "tab:orange", "tab:green"]):
    scaled_boundary = (scale * w_geometry[0] * horizontal + scale * b_geometry) / (
        -scale * w_geometry[1]
    )
    axes[0].plot(horizontal, scaled_boundary, color=color, linewidth=2,
                 label=rf"$c={scale:g}$")
axes[0].scatter(*q, color="tab:red", s=55, zorder=4)
axes[0].set(title="Rescaling parameters leaves the boundary fixed",
            xlabel=r"$x_1$", ylabel=r"$x_2$", xlim=(-0.5, 2.8), ylim=(-2, 4))
axes[0].legend(title=r"$(cw,cb)$", fontsize=8)
axes[1].plot(scales, functional_margins, "o-", linewidth=2.2,
             label="functional margin")
axes[1].plot(scales, geometric_margins, "s--", linewidth=2.2,
             label="geometric margin")
axes[1].set(title="Only normalized distance is scale invariant",
            xlabel=r"positive parameter scale $c$", ylabel="margin value")
axes[1].legend()
plt.tight_layout()

assert np.allclose(geometric_margins, geometric_margins[0])
assert np.allclose(functional_margins, scales * functional_margins[1])

### 2.3 Canonical scaling turns separation into margin maximization

Suppose the training data are linearly separable. Because positive rescaling changes functional margins but not the boundary, choose the convenient **canonical scaling**

$$\min_i y_i(w^\top x_i+b)=1.$$

The nearest observations then touch $z=+1$ or $z=-1$. Each supporting hyperplane lies $1/\lVert w\rVert_2$ from $z=0$, so the full margin width is

$$\frac{1}{\lVert w\rVert_2}+\frac{1}{\lVert w\rVert_2}=\frac{2}{\lVert w\rVert_2}.$$

Maximizing this width is equivalent to minimizing $\lVert w\rVert_2$, conventionally written as the convex hard-margin problem

$$\min_{w,b}\ \frac12\lVert w\rVert_2^2\quad\text{subject to}\quad y_i(w^\top x_i+b)\ge1\quad\text{for every }i.$$

The constraints prevent the meaningless solution $w=0$. The following diagram uses the same $w$ and $b$ as above so that the algebra and picture refer to one object.

In [ ]:
toy_points = np.array([[0.5, 1.0], [0.0, 0.5], [0.2, 1.2],
                       [1.5, 1.0], [2.2, 1.3], [1.6, 0.5]])
toy_labels = np.array([-1, -1, -1, 1, 1, 1])
toy_scores = toy_points @ w_geometry + b_geometry
toy_functional_margins = toy_labels * toy_scores
margin_width = 2.0 / np.linalg.norm(w_geometry)

fig, axis = plt.subplots(figsize=(8, 6))
for label, color in [(-1, "tab:blue"), (1, "tab:red")]:
    mask = toy_labels == label
    axis.scatter(toy_points[mask, 0], toy_points[mask, 1], s=75, color=color,
                 label=f"class {label:+d}", zorder=4)
for level, style, width in [(-1.0, "--", 1.8), (0.0, "-", 2.6), (1.0, "--", 1.8)]:
    vertical = (level - b_geometry - w_geometry[0] * horizontal) / w_geometry[1]
    axis.plot(horizontal, vertical, style, color="black", linewidth=width,
              label=rf"$z={level:g}$")
support_points = toy_points[np.isclose(toy_functional_margins, 1.0)]
axis.scatter(support_points[:, 0], support_points[:, 1], s=130,
             facecolors="none", edgecolors="tab:orange", linewidths=2,
             label="support vector", zorder=5)
# Draw the margin width along the normal through one boundary point.
boundary_point = -b_geometry * w_geometry / (w_geometry @ w_geometry)
unit_normal = w_geometry / np.linalg.norm(w_geometry)
negative_edge = boundary_point - unit_normal / np.linalg.norm(w_geometry)
positive_edge = boundary_point + unit_normal / np.linalg.norm(w_geometry)
axis.annotate("", xy=positive_edge, xytext=negative_edge,
              arrowprops={"arrowstyle": "<->", "lw": 2.2,
                          "color": "tab:green"})
axis.text(0.15, -0.65, rf"margin width $2/\|w\|={margin_width:.3f}$",
          color="tab:green", fontsize=10,
          bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"})
axis.set(title="Canonical scaling makes the closest points support the margin",
         xlabel=r"$x_1$", ylabel=r"$x_2$", xlim=(-0.5, 2.8),
         ylim=(-2.0, 4.0), aspect="equal")
axis.legend(ncols=2, fontsize=8, loc="lower right")
plt.tight_layout()

assert np.all(toy_functional_margins >= 1.0)
assert np.isclose(toy_functional_margins.min(), 1.0)

### Read the picture as an optimization problem

The three black lines are not three classifiers. They are parallel level sets of one score: $z=0$ is the decision boundary, while $z=-1$ and $z=+1$ mark the edges of the margin. The empty band between the dashed lines has width $2/\lVert w\rVert_2$.

The gold-ringed observations prevent this band from widening further without changing the boundary or violating a constraint. Those closest observations therefore **support** the solution. The hard-margin optimization searches over both the orientation $w$ and offset $b$ to find the widest feasible band, not merely a line that classifies the training points correctly.

## 3. Worked example: carry the geometry into a real dataset

The toy construction made every distance hand-computable. We now ask whether the same objects remain interpretable after fitting real measurements. We use two image-derived variables from the repository’s Wisconsin breast-cancer table because two dimensions let us see the learned boundary, normal direction, supporting levels, and margin violations directly. This is an instructional geometry example, not a clinical device. Labels are explicitly recoded to $\{-1,+1\}$.

In [ ]:
database_uri = f"{course_database_path().as_uri()}?mode=ro"
query = (
    "SELECT observation_id, mean_radius, mean_texture, "
    "diagnosis_code, diagnosis_label "
    "FROM breast_cancer_observations ORDER BY observation_id"
)
with sqlite3.connect(database_uri, uri=True) as connection:
    cancer = pd.read_sql_query(query, connection)

features = cancer[["mean_radius", "mean_texture"]]
signed_target = np.where(cancer["diagnosis_code"].to_numpy() == 1, 1, -1)
display(cancer.head())
display(cancer.groupby(["diagnosis_code", "diagnosis_label"]).size().rename("count"))
display(features.describe().T)
assert cancer["observation_id"].is_unique
assert not cancer.isna().any().any()
assert set(signed_target) == {-1, 1}

### Scaling is part of the margin definition

The penalty $\lVert w\rVert_2^2$ depends on feature units. We split first, fit scaling only on training observations, and reuse that fitted state on test observations. Scaling before splitting would leak distribution information.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, signed_target, test_size=0.25,
    stratify=signed_target, random_state=RANDOM_STATE,
)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = LinearSVM(
    regularization_strength=0.01,
    learning_rate=0.02,
    epochs=4_000,
).fit(X_train_scaled, y_train)

print("weights:", svm.weights_)
print("bias:", svm.bias_)
print("support candidates:", int(svm.support_mask_.sum()))
assert svm.weights_.shape == (2,)
assert svm.loss_history_[-1] < svm.loss_history_[0]

### Read fitted parameters as geometry

The fitted vector $w=(w_1,w_2)$ points normal to the decision boundary. The equation $w_1x_1+w_2x_2+b=c$ traces a line of constant score $c$. Therefore the three levels $c=-1,0,+1$ are parallel: $c=0$ is the decision boundary and $c=\pm1$ are the canonical margin levels. Their perpendicular separation is $2/\lVert w\rVert_2$.

The orange circles below mark training observations with $y_i z_i\le1$. They are either on the margin, inside it, or misclassified. These are the observations whose hinge terms remain active at the fitted parameters.

In [ ]:
fitted_margin_width = 2.0 / np.linalg.norm(svm.weights_)
fig, axis = plt.subplots(figsize=(8, 6))
for label, color in [(-1, "tab:blue"), (1, "tab:red")]:
    mask = y_train == label
    axis.scatter(X_train_scaled[mask, 0], X_train_scaled[mask, 1],
                 s=22, alpha=0.48, color=color, label=f"class {label:+d}")

support = X_train_scaled[svm.support_mask_]
axis.scatter(support[:, 0], support[:, 1], s=78, facecolors="none",
             edgecolors="tab:orange", linewidths=1.2, label="on/inside margin")
x_limits = np.quantile(X_train_scaled[:, 0], [0.01, 0.99]) + np.array([-0.35, 0.35])
y_limits = np.quantile(X_train_scaled[:, 1], [0.01, 0.99]) + np.array([-0.45, 0.45])
horizontal = np.linspace(*x_limits, 300)
for level, style, width in [(-1.0, "--", 1.4), (0.0, "-", 2.4), (1.0, "--", 1.4)]:
    vertical = (level - svm.bias_ - svm.weights_[0] * horizontal) / svm.weights_[1]
    axis.plot(horizontal, vertical, style, color="black", linewidth=width,
              label=f"score = {level:g}")
axis.set(title=rf"Learned boundary and geometric margin (width $={fitted_margin_width:.2f}$)",
         xlabel="standardized mean radius", ylabel="standardized mean texture",
         xlim=x_limits, ylim=y_limits)
axis.legend(ncols=2, fontsize=8)
plt.tight_layout()
assert np.isfinite(vertical).all()

## 4. From impossible constraints to soft margins

The hard-margin constraints $y_i z_i\ge1$ fail as soon as the classes overlap. Rather than abandon the geometry, introduce one nonnegative slack variable $\xi_i$ per observation:

$$y_i z_i\ge1-\xi_i,\qquad \xi_i\ge0.$$

For fixed $w$ and $b$, the smallest feasible slack is

$$\xi_i=\max(0,1-y_i z_i).$$

This formula records geometry in three regions:

| signed margin $y_i z_i$ | geometric status | smallest slack / hinge loss |
|---|---|---|
| $y_i z_i>1$ | correct and beyond the margin | $0$ |
| $0<y_i z_i<1$ | correct but inside the margin | $1-y_i z_i$ |
| $y_i z_i\le0$ | on the boundary or misclassified | at least $1$ |

Penalizing the smallest slacks produces hinge loss. Combining it with the width preference gives the objective used in this notebook:

$$
J(w,b)=\frac{\lambda}{2}\lVert w\rVert_2^2
+\frac1n\sum_{i=1}^n\max\bigl(0,1-y_i(w^\top x_i+b)\bigr).
$$

The first term prefers a small $\lVert w\rVert_2$ and therefore a wide geometric margin. The second term pays for violations of the desired unit margin. The optimizer must negotiate this explicit tradeoff; the bias is not penalized here.

A common alternative writes $\frac12\lVert w\rVert^2+C\sum_i\xi_i$. Larger $C$ penalizes violations more; larger $\lambda$ in our convention regularizes more. Exact conversion depends on summed versus averaged loss, so every implementation must document its convention. The plot below should now read as a picture of margin violation, not as an arbitrary piecewise formula.

In [ ]:
signed_margin_grid = np.linspace(-2.0, 2.5, 501)
hinge_loss = np.maximum(0.0, 1.0 - signed_margin_grid)

fig, axis = plt.subplots(figsize=(9, 4.5))
axis.axvspan(-2.0, 0.0, color="tab:red", alpha=0.10, label="misclassified")
axis.axvspan(0.0, 1.0, color="tab:orange", alpha=0.14,
             label="correct, inside margin")
axis.axvspan(1.0, 2.5, color="tab:green", alpha=0.10,
             label="correct, beyond margin")
axis.plot(signed_margin_grid, hinge_loss, color="black", linewidth=2.8,
          label=r"hinge loss $\max(0,1-yz)$")
example_margins = np.array([-0.6, 0.4, 1.0, 1.7])
axis.scatter(example_margins, np.maximum(0.0, 1.0 - example_margins),
             color="tab:blue", s=55, zorder=4)
axis.axvline(0.0, color="0.5", linestyle=":", label="decision boundary")
axis.axvline(1.0, color="tab:green", linestyle="--", label="unit margin")
axis.set(xlabel=r"signed functional margin $y_i z_i$", ylabel="hinge loss / minimum slack",
         ylim=(-0.05, 3.2), title="Hinge loss measures violation of the unit-margin requirement")
axis.legend(ncols=2, fontsize=8)
plt.tight_layout()
assert hinge_loss[np.argmin(np.abs(signed_margin_grid - 1.5))] == 0.0
assert np.isclose(np.maximum(0.0, 1.0 - 0.4), 0.6)

## 5. Subgradients and support vectors

Hinge loss is convex but not differentiable at $y_i z_i=1$. Away from that kink, one score subgradient is

$$
\frac{\partial \ell_i}{\partial z_i}=
\begin{cases}
-y_i, & y_i z_i<1,\\
0, & y_i z_i>1.
\end{cases}
$$

Thus

$$
g_w=\lambda w-\frac1n\sum_{i:\,y_i z_i<1}y_i x_i,
\qquad
g_b=-\frac1n\sum_{i:\,y_i z_i<1}y_i.
$$

Observations beyond the margin contribute zero local hinge subgradient. Points on or within the margin anchor the solution and motivate the name **support vector**.

In [ ]:
toy_scores = np.array([-0.5, 0.4, 1.0, 1.8])
toy_targets = np.ones(4)
toy_margins = toy_targets * toy_scores
active = toy_margins < 1.0
chosen_subgradient = np.where(active, -toy_targets, 0.0)
display(pd.DataFrame({
    "score z": toy_scores,
    "signed margin yz": toy_margins,
    "inside unit margin": active,
    "chosen d hinge / dz": chosen_subgradient,
}))
assert np.array_equal(chosen_subgradient, [-1.0, -1.0, 0.0, 0.0])

## 6. Held-out evidence and output semantics

A linear SVM returns a signed score and class label. Its score is not a probability. Balanced accuracy and a confusion matrix evaluate decisions; calibration metrics require a separately fitted probability-calibration procedure and appropriate held-out data.

In [ ]:
test_predictions = svm.predict(X_test_scaled)
balanced_accuracy = balanced_accuracy_score(y_test, test_predictions)
print(f"held-out balanced accuracy: {balanced_accuracy:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(svm.loss_history_)
axes[0].set(title="Optimization trace", xlabel="update",
            ylabel="regularized hinge objective", yscale="log")
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, display_labels=["class -1", "class +1"],
    ax=axes[1], colorbar=False,
)
axes[1].set_title("Held-out decisions")
plt.tight_layout()
assert 0.5 <= balanced_accuracy <= 1.0

## 7. One computation, three learning problems

| Model | Score | Training loss | Characteristic output |
|---|---|---|---|
| Identity single neuron (linear regression) | $w^\top x+b$ | squared error | real-valued prediction |
| Sigmoid single neuron (logistic regression) | $w^\top x+b$ | log loss | conditional probability |
| Identity single neuron (linear SVM) | $w^\top x+b$ | regularized hinge | score and signed class |

The shared mathematical mechanism is the affine score. The learning problem changes when we change the activation, loss, regularization, or interpretation of the output. A kernel SVM is different again: it is linear in kernel features but generally nonlinear in the original input space.

## Debugging and common failure modes

- Convert labels explicitly to $-1$ and $+1$ before computing $y z$.
- Fit scaling after splitting because the norm penalty depends on units.
- State whether the bias is penalized and whether loss is summed or averaged.
- Never call an uncalibrated SVM score a probability.
- Remember that larger $C$ and larger $\lambda$ move common conventions in opposite directions.
- Preserve held-out data; training hinge loss is not release evidence.

Inspect label coding, feature statistics, signed margins, active-constraint counts, objective history, and finite values in that order.

## Professional practice

| Data scientist asks | Software engineer asks |
|---|---|
| Is a linear boundary plausible? | Are feature order and scaling state versioned? |
| Which mistakes carry more cost? | Is decision policy separate from score computation? |
| How sensitive is the result to regularization? | Is the exact convention serialized? |
| Does the score need probability meaning? | Is calibration a separate fitted component? |
| Which observations anchor the margin? | Are support-vector attributes inspectable? |

In [ ]:
# Guided practice: turn signed margins into diagnostic categories.
def margin_status(signed_margin: np.ndarray) -> np.ndarray:
    result = np.full(signed_margin.shape, "beyond margin", dtype=object)
    result[signed_margin < 1.0] = "inside margin"
    result[signed_margin <= 0.0] = "misclassified or boundary"
    return result

practice_margins = np.array([-0.4, 0.3, 1.0, 1.7])
practice_status = margin_status(practice_margins)
print(practice_status)
assert practice_status[0] == "misclassified or boundary"
assert practice_status[1] == "inside margin"
assert practice_status[-1] == "beyond margin"

## Practice ladder

### Guided practice
Derive the distance between $z=+1$ and $z=-1$. **Success criterion:** obtain $2/\lVert w\rVert_2$ and explain the normalization.

### Independent practice
Compare three regularization strengths using training and validation data only. Plot margin width, active-constraint count, and validation balanced accuracy.

### Extension
Check a hinge subgradient away from the kink and explain why an ordinary derivative is undefined exactly at $y z=1$.

### Contribution studio
Add a tested regularized hinge objective or calibration adapter. Document label domain, aggregation, bias policy, score semantics, and failures.

## Retrieval practice, takeaway, and further reading

1. What mathematical object is shared by logistic regression and a linear SVM?
2. Why does sharing a hyperplane not make the estimators equivalent?
3. Why is hard-margin width $2/\lVert w\rVert_2$?
4. Which observations contribute nonzero hinge subgradients?
5. Why must scaling be fitted after splitting?
6. Why is an SVM score not automatically a probability?

**Takeaway:** a linear SVM is an affine-score classifier trained by a maximum-margin objective. Its relationship to a single neuron is structural, not an equivalence of losses, semantics, or algorithms.

Next, Lecture 12 Notebook 00 will use linear regression, logistic regression, and this SVM to motivate a reusable software design.

**Further reading:** [Python data model](https://docs.python.org/3/reference/datamodel.html), scikit-learn’s SVM user guide, and Cortes and Vapnik (1995), “Support-Vector Networks.”